In [1]:
import pandas as pd
import numpy as np

DATA_FOLDER = "./"
df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_big_best_customers_4c.pickle")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()
df = df[df["product_id"].isin(product_ids)]
df = df.sort_values(by=["date_id", "product_id"])
df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)


/tmp/ipykernel_45982/2806736901.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)


In [2]:
df.describe()

,product_id,customer_id,periodo_min_producto,periodo_max_producto,periodo_min_customer,periodo_max_customer,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,...,prod_tn_rolling_mean_24_x_tn_rolling_mean_12_lag_2,prod_tn_rolling_mean_24_x_tn_rolling_mean_24_lag_1,prod_tn_rolling_mean_24_x_tn_rolling_mean_12_lag_3,prod_tn_wavelet_0_mean_lag_6_x_tn_rolling_mean_12_lag_2,prod_tn_wavelet_0_mean_lag_6_x_tn_rolling_mean_24_lag_1,prod_tn_wavelet_0_mean_lag_6_x_tn_rolling_mean_12_lag_3,prod_tn_rolling_mean_12_lag_2_x_tn_rolling_mean_24_lag_1,prod_tn_rolling_mean_12_lag_2_x_tn_rolling_mean_12_lag_3,prod_tn_rolling_mean_24_lag_1_x_tn_rolling_mean_12_lag_3,target
count,111875.000000,111875.000000,89500,89500,89500,89500,89500.000000,111875.000000,111875.000000,111875.000000,...,34275.000000,31435.000000,34275.000000,64620.000000,31435.000000,61340.000000,31435.000000,61340.000000,31435.000000,104075.000000
mean,20479.289698,8002.000000,2017-04-03 16:16:45.264804352,2019-12-01 00:00:00.000000256,2017-01-01 00:00:00,2019-12-01 00:00:00.000000256,0.009397,44.502177,10.260865,10.034459,...,2110.056885,2232.796387,2132.908936,2083.338135,2212.311035,2107.915039,2119.750000,2185.477051,2140.454346,10.174339
min,20001.000000,0.000000,2017-01-01 00:00:00,2019-12-01 00:00:00,2017-01-01 00:00:00,2019-12-01 00:00:00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,20200.000000,10001.000000,2017-01-01 00:00:00,2019-12-01 00:00:00,2017-01-01 00:00:00,2019-12-01 00:00:00,0.000000,2.000000,0.061150,0.061150,...,0.035324,0.042236,0.036213,0.035096,0.043890,0.036370,0.035432,0.034931,0.036262,0.065520
50%,20411.000000,10002.000000,2017-01-01 00:00:00,2019-12-01 00:00:00,2017-01-01 00:00:00,2019-12-01 00:00:00,0.000000,7.000000,0.594050,0.591770,...,0.746867,0.800221,0.754865,0.715975,0.824083,0.727980,0.752381,0.716410,0.761213,0.606060
75%,20730.000000,10003.000000,2017-01-01 00:00:00,2019-12-01 00:00:00,2017-01-01 00:00:00,2019-12-01 00:00:00,0.000000,21.000000,3.609775,3.570840,...,21.664049,23.673111,21.998564,21.205910,23.623253,21.600277,21.751461,21.952288,22.180233,3.616700
max,21276.000000,10004.000000,2019-09-01 00:00:00,2019-12-01 00:00:00,2017-01-01 00:00:00,2019-12-01 00:00:00,1.000000,682.000000,1575.216919,1493.351929,...,848909.312500,800085.437500,861155.625000,812882.937500,765096.375000,812882.937500,855557.250000,901930.625000,848909.312500,1493.351929
std,334.546877,4001.018007,NaN,NaN,NaN,NaN,0.096480,89.899754,46.760345,45.223530,...,23537.558594,23617.437500,23576.576172,22610.666016,23212.599609,22723.259766,23587.189453,23861.630859,23640.666016,45.976662


In [3]:
df[["fecha", "date_id"]]

,fecha,date_id
0,2017-01,0
36,2017-01,0
72,2017-01,0
108,2017-01,0
144,2017-01,0
...,...,...
157374,2019-12,35
157384,2019-12,35
157394,2019-12,35
157404,2019-12,35


In [4]:
# transformacion comun de datos para todos los modelos:
# remuevo periodo_min_producto, periodo_max_producto, periodo_min_customer, periodo_max_customer
df = df.drop(columns=["periodo_min_producto", "periodo_max_producto",
                   "periodo_min_customer", "periodo_max_customer"], errors='ignore')

# transformo columnas object a categorical
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype("category")

# transformo plan precios cuidados a categorical
df["plan_precios_cuidados"] = df["plan_precios_cuidados"].astype("category")

In [5]:

# dropeo columns donde tenga mas sea todo nan hasta date_id 28
print(f"Df shape before dropping columns: {df.shape}")
subset = df[df["date_id"] <= 28]
cols_to_drop = subset.columns[subset.isna().all()]
df = df.drop(columns=cols_to_drop, errors='ignore')
print(f"Df shape after dropping columns: {df.shape}")

Df shape before dropping columns: (111875, 759)
Df shape after dropping columns: (111875, 715)


In [6]:

from sklearn.model_selection import BaseCrossValidator
import numpy as np

class CustomTimeSeriesSplit(BaseCrossValidator):
    def __init__(self, n_splits=3, gap=1):
        self.n_splits = n_splits
        self.gap = gap

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        # Asegurar que X es DataFrame
        
        unique_dates = sorted(X["date_id"].unique(), reverse=True)
        
        for i in range(self.n_splits):
                
            test_date_id = unique_dates[i]
            train_date_id = test_date_id - self.gap - 1
            
            # Usar np.where para obtener posiciones enteras
            train_mask = X["date_id"] <= train_date_id
            test_mask = X["date_id"] == test_date_id
            
            train_idx = np.where(train_mask)[0]
            test_idx = np.where(test_mask)[0]
            
            yield train_idx, test_idx

In [7]:

class LinearRegressionModel:
    def __init__(self):
        self.model = None
    
    @property
    def name(self):
        return "LinearRegression"
    
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum"}).sort_values(["product_id", "date_id"])
        for lag in range(1, 12):
            df[f"tn_{lag}"] = df.groupby("product_id")["tn"].shift(lag)
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):
        from sklearn.linear_model import LinearRegression
        # si quiero  201912, entreno con 201812 (por estacionalidad)
        # lo busco dinamicamente con pred_df
        date_id_pred = pred_df["date_id"].unique()[0]
        train_df = train_df[train_df["date_id"] == (date_id_pred-12)]
        prod_ids_magicos = [20002, 20003, 20006, 20010, 20011, 20018, 20019, 20021,
            20026, 20028, 20035, 20039, 20042, 20044, 20045, 20046, 20049,
            20051, 20052, 20053, 20055, 20008, 20001, 20017, 20086, 20180,
            20193, 20320, 20532, 20612, 20637, 20807, 20838]
        train_df = train_df[train_df["product_id"].isin(prod_ids_magicos)]

        # elimino registros incompletos
        features = ["tn"] + [f"tn_{lag}" for lag in range(1, 12)]
        target = "target"
        train_df = train_df.dropna(subset=features + [target])
        print(f"Registros de entrenamiento: {len(train_df)}")
        X = train_df[features]
        y = train_df[target]
        model = LinearRegression()
        model.fit(X, y)
        self.model = model

        # hago la prediccion
        pred_df = pred_df.copy()
        
        # solo hace la prediccion para los productos que tienen todas las features
        pred_df = pred_df.dropna(subset=features)
        X_pred = pred_df[features]
        pred_df["prediction"] = model.predict(X_pred).clip(min=0)  # Aseguro que la prediccion no sea negativa
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })
    

In [8]:
class SimpleMovingAveragePredictor:
    ''' Usa una media movil simple para predecir tn'''
    def __init__(self, window_size=12):
        self.model = None
        self.window_size = window_size

    @property
    def name(self):
        return f"SMA-{self.window_size}"
    
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum"}).sort_values(["product_id", "date_id"])
        # hago una columna que es la media movil simple agrupada por producto
        df["tn_sma"] = df.groupby("product_id")["tn"].transform(
            lambda x: x.rolling(window=self.window_size, min_periods=1).mean()
        )
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):

        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["tn_sma"],
        })

In [9]:
class ExponentialMovingAveragePredictor:
    ''' Usa una media movil exponencial para predecir tn'''
    def __init__(self, window_size=12):
        self.model = None
        self.window_size = window_size

    @property
    def name(self):
        return f"EMA-{self.window_size}"
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum"}).sort_values(["product_id", "date_id"])
        # hago una columna que es la media movil exponencial agrupada por producto
        df["tn_ema"] = df.groupby("product_id")["tn"].transform(
            lambda x: x.ewm(span=self.window_size, adjust=False).mean()
        )
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["tn_ema"],
        })

In [10]:
class AutoGluonPredictor:
    def __init__(self, presets="best_quality", estimator=None):
        self.model = None
        self.presets = presets
        self.estimator = estimator

    @property
    def name(self):
        if self.estimator:
            return f"AutoGluon-{self.estimator}"
        return f"AutoGluon-{self.presets}"
    
    def prepare_dataset(self, df):

        df = df.copy()
        df["fecha"] = df["fecha"].apply(lambda x: x.to_timestamp("M"))
        df = df.rename(columns={"fecha": "timestamp"})
        df["product_id"] = df["product_id"].astype(int)
        df["serie_id"] = df["product_id"].astype(str) + "-" + df["customer_id"].astype(str)
        df["cat1"] = df["cat1"].astype("category")
        df["cat2"] = df["cat2"].astype("category")
        df["cat3"] = df["cat3"].astype("category")
        df["brand"] = df["brand"].astype("category")
        df["sku_size"] = df["sku_size"].astype("category")

        self.static_features_df = pd.DataFrame({
            "cat1": df.groupby("serie_id")["cat1"].first(),
            "cat2": df.groupby("serie_id")["cat2"].first(),
            "cat3": df.groupby("serie_id")["cat3"].first(),
            "brand": df.groupby("serie_id")["brand"].first(),
            "sku_size": df.groupby("serie_id")["sku_size"].first(),
            "customer_id": df.groupby("serie_id")["customer_id"].first(),
            "product_id": df.groupby("serie_id")["product_id"].first(),
        }).reset_index()
        

        min_periods = 12  # Mínimo 6 meses de datos
        product_counts = df.groupby(["product_id"]).size()
        valid_products = product_counts[product_counts >= min_periods].index
        df = df[df["product_id"].isin(valid_products)]        
        df = df.dropna(subset=["tn"])
        return df
    
    def fit_and_predict(self, train_df, pred_df, df_model):
        from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
        # el autogluon lo entreno con todas las fechas hasta pred_df
        train_df = df_model[df_model["date_id"] <= pred_df["date_id"].unique()[0]]
        ts_data = TimeSeriesDataFrame.from_data_frame(
            train_df.drop(columns=["target"]), 
            id_column="serie_id", 
            timestamp_column="timestamp", 
            static_features_df=self.static_features_df
        )
        ts_data = ts_data.sort_index()
        ts_data = ts_data.fill_missing_values()

        predictor = TimeSeriesPredictor(
            prediction_length=2,
            target="tn",
            freq="MS",
        )
        if self.estimator:
            predictor.fit(ts_data, hyperparameters={self.estimator: {}})
        else:
            predictor.fit(ts_data, presets=self.presets)
        forecast = predictor.predict(ts_data)
        forecast_mean = forecast["mean"].reset_index()
        forecast_mean = forecast_mean[forecast_mean["timestamp"] == forecast_mean["timestamp"].max()]
        forecast_mean[["product_id", "customer_id"]] = forecast_mean["item_id"].str.split("-", expand=True)
        forecast_mean["product_id"] = forecast_mean["product_id"].astype(int)
        forecast_mean = forecast_mean.groupby("product_id").agg({
            "mean": "sum",
        }).reset_index()

        # rename item_id to product_id
        pred_df = pred_df.copy()
        pred_df["product_id"] = pred_df["product_id"].astype(int)
        # separo serie_id en product_id y customer_id
        pred_df = pred_df.groupby(["product_id"]).agg({
            "target": "sum",
            "date_id": "first"
        }).reset_index()
        pred_df = pred_df.merge(forecast_mean, on=["product_id"], how="left")
        pred_df = pred_df.rename(columns={"mean": "prediction"})
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })



In [ ]:


class BaseTabularPredictor:
    
    def _scaling_df(self, df, train=True):
        df = df.copy()
        import re
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        transformations = {
            "tn": [
                r"tn$",
                r"cust_request_qty_per_tn$",
                r"tn_lag_*",
                r"tn_rolling_mean_*",
                r"tn_rolling_max_*",
                r"tn_rolling_min_*",
                r"tn_.*_vendidas$",
                r"tn_agg*",
                r"tn_wavelet_*",
            ]
            + [r"stock_final$"]
            + [r"cust_request_tn_minus_tn$"]
            + [r"tn_diff_*"],
            "cust_request_qty": [
                r"cust_request_qty$",
                r"cust_request_qty_lag_*",
                r"cust_request_qty_rolling_mean_*",
                r"cust_request_qty_rolling_max_*",
                r"cust_request_qty_rolling_min_*",
                r"cust_request_qty_.*_vendidas$",
                r"cust_request_qty_agg*",
                r"cust_request_qty_wavelet_*",
            ]
            + [r"cust_request_qty_diff_*"],
        }

        # busco todas las columnas que empiezan con prod_ y agrego key y valor en transformation
        for col in numeric_cols:
            if col.startswith("prod_"):
                transformations[col] = [r"{}$".format(col)]

        from pandas.errors import PerformanceWarning
        import warnings
        warnings.simplefilter(action="ignore", category=PerformanceWarning)
        df = df.set_index(['serie_id', "date_id"])
        if train:
            prod_stats = df.groupby(["serie_id"])[
                list(transformations.keys())
            ].agg(["std"])
            prod_stats.columns = [
                f"{col[0]}_{col[1]}" for col in prod_stats.columns
            ]  # renombro las columnas para que no tengan tupla

            prod_stats = prod_stats.reset_index()
            self.prod_stats = prod_stats
            prod_stats = prod_stats.set_index(['serie_id'])
            # supress performance warnings
            self.prod_stats = prod_stats

            print("Scaling")
        else:
            if self.prod_stats is None:
                raise ValueError("prod_stats is not set. Call prepare_dataset first.")
            prod_stats = self.prod_stats
        for trainer, regex_cols in transformations.items():
            for col in regex_cols:
                matching_cols = [c for c in numeric_cols if re.match(col, c)]
                if not matching_cols:
                    continue
                for col in matching_cols:
                    std_col = prod_stats[trainer + "_std"]
                    df[f"{col}_scaled"] = (df[col] / std_col).replace([np.inf, -np.inf], np.nan)

        # scalo el target con tn_std
        df["target_scaled"] = df["target"] / prod_stats["tn_std"]
        df["target_scaled"] = df["target_scaled"].replace([np.inf, -np.inf], np.nan).fillna(0)

        df = df.reset_index()
        return df

    def prepare_dataset(self, df):
        df = df.copy()
        df["serie_id"] = df["product_id"].astype(str) + "-" + df["customer_id"].astype(str)
        return df


class AutoMLPredictor(BaseTabularPredictor):
    
    def __init__(self, estimator="lgbm", time_budget=60):
        self.model = None
        self.prod_stats = None
        self.estimator = estimator
        self.time_budget = time_budget

    @property
    def name(self):
        return f"AutoML-{self.estimator}-{self.time_budget}s"
    
    def custom_metric(self, X_val, y_val, estimator, labels, X_train, y_train, *args, **kwargs):
        y_pred = estimator.predict(X_val)
    
        temp_df = pd.DataFrame({
            "product_id": X_val["product_id"].values,
            "customer_id": X_val["customer_id"].values,
            "y_true": y_val,
            "y_pred": y_pred
        })
        temp_df["product_id"] = temp_df["product_id"].astype(int)
        temp_df["customer_id"] = temp_df["customer_id"].astype(int)
        prod_stats = self.prod_stats.copy().reset_index()
        prod_stats[["product_id", "customer_id"]] = prod_stats["serie_id"].str.split("-", expand=True)
        prod_stats["product_id"] = prod_stats["product_id"].astype(int)
        prod_stats["customer_id"] = prod_stats["customer_id"].astype(int)
        temp_df = temp_df.merge(prod_stats[["product_id", "customer_id", "tn_std"]], on=["product_id", "customer_id"], how="left")
        # desescale the predictions
        temp_df["y_pred"] = temp_df["y_pred"] * temp_df["tn_std"]
        temp_df["y_true"] = temp_df["y_true"] * temp_df["tn_std"]
    
        grouped = temp_df.groupby("product_id")[["y_true", "y_pred"]].sum()
        total_true = grouped["y_true"].sum()
    
        if total_true == 0:
            return 0.0, {"total_error": 0.0}
    
        total_error = np.abs(grouped["y_pred"] - grouped["y_true"]).sum() / total_true
        return total_error, {"total_error": total_error}

    def fit_and_predict(self, train_df, pred_df, df_model):
        from flaml import AutoML
        train_df = self._scaling_df(train_df, train=True)
        pred_df = self._scaling_df(pred_df, train=False)
        tscv = CustomTimeSeriesSplit(2, gap=1)
        automl_settings = {
            "time_budget": self.time_budget,
            "task": "regression",
            "metric": self.custom_metric,
            "estimator_list": [self.estimator],
            "n_jobs": -1,
            "eval_method": "cv",
            "split_type": tscv,
            "verbose": 3,
            "retrain_full": True
        }
        X_train = train_df.drop(columns=["target", "target_scaled", "fecha"])
        y_train = train_df["target_scaled"]
        automl = AutoML()
        automl.fit(X_train, y_train, **automl_settings)
        # hago la prediccion
        y_pred = automl.predict(pred_df.drop(columns=["target", "target_scaled", "fecha"]))
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()
 
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })
        


In [ ]:
class AutoGluonTabularPredictor(BaseTabularPredictor):
    
    def __init__(self, presets="medium_quality", exclude_model_types=None, time_budget=None, subsample=1):
        self.model = None
        self.prod_stats = None
        self.presets = presets
        self.exclude_model_types = exclude_model_types or []
        self.time_budget = time_budget
        self.subsample = subsample

    @property
    def name(self):
        return f"AutoGluonTabular-{self.presets}-budget-{self.time_budget}s-subsample-{self.subsample}"
    
    def fit_and_predict(self, train_df, pred_df, df_model):
        from autogluon.tabular import TabularPredictor, TabularDataset
        train_df = self._scaling_df(train_df, train=True)
        pred_df = self._scaling_df(pred_df, train=False)

        # uso el date_id mas alto de train_df como tunning_data
        tunning_df = train_df[train_df["date_id"] == train_df["date_id"].max()]
        train_df = train_df[train_df["date_id"] < train_df["date_id"].max()]
        train_df = train_df.sample(frac=self.subsample, random_state=42)
        train = TabularDataset(train_df.drop(columns=["fecha"]))
        tunning = TabularDataset(tunning_df.drop(columns=["fecha"]))
        pred = TabularDataset(pred_df.drop(columns=["fecha"]))

        predictor = TabularPredictor(
            label="target_scaled",
            eval_metric="mean_absolute_error",
        )
        predictor.fit(
            train.drop(columns=["target"]),
            presets=self.presets,
            tuning_data=tunning.drop(columns=["target"]),
            excluded_model_types=["RF", "XT"] + self.exclude_model_types,
            time_limit=self.time_budget,
        )

        y_pred = predictor.predict(pred.drop(columns=["target", "target_scaled"]))
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()
 
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })

In [13]:
#autogluon_tabular_predictor = AutoGluonTabularPredictor(presets="medium")
#df_autogluon_tabular = autogluon_tabular_predictor.prepare_dataset(df)
#test_df = df_autogluon_tabular[df_autogluon_tabular["date_id"] == 33]
#train_df = df_autogluon_tabular[df_autogluon_tabular["date_id"] < 32]
#results = autogluon_tabular_predictor.fit_and_predict(train_df, test_df, df_autogluon_tabular)

In [14]:

# import deep copy
from copy import deepcopy
class EnsambleTrainer:
    def __init__(self, models):
        self.models = models
        self.model_weights = None
        self.train_results = None

    def _combina_results(self, results):
        from collections import defaultdict
        # Diccionario para almacenar resultados intermedios
        combined_results = defaultdict(dict)

        for split, models in results.items():
            for model_info in models:
                model_name = model_info["model"].name
                pred_df = model_info["pred_df"]

                for _, row in pred_df.iterrows():
                    key = (row["product_id"], row["date_id"])
                    combined_results[key]["product_id"] = row["product_id"]
                    combined_results[key]["date_id"] = row["date_id"]
                    combined_results[key]["target"] = row["target"]
                    combined_results[key][f"prediction_{model_name}"] = row["prediction"]

        # Convertir a DataFrame
        results_df = pd.DataFrame(combined_results.values())

        # Opcional: ordenar columnas
        cols = ["product_id", "date_id", "target"] + sorted([col for col in results_df.columns if col not in {"product_id", "date_id", "target"}])
        results_df = results_df[cols]

        def fill_row_na_with_row_mean(row, prediction_cols):
            preds = row[prediction_cols]
            row[prediction_cols] = preds.fillna(preds.mean(skipna=True))
            return row

        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]

        results_df = results_df.apply(fill_row_na_with_row_mean, axis=1, prediction_cols=prediction_cols)   
        self.train_results = results_df
        return results_df
        
    def _optimize_weights(self, results_df):
        # Asegurar columnas de predicción con prefijo prediction_
        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]

        # Lista donde irán los dicts de pesos y predicciones agregadas
        weights_list = []
        predictions_list = []

        # Para guardar pesos temporales por product_id
        product_weights = {}

        # Por cada fila (product_id, date_id), elegimos el mejor modelo (peso 1 para ese, 0 para el resto)
        for _, row in results_df.iterrows():
            product_id = row["product_id"]
            target = row["target"]

            preds = np.array([row[col] for col in prediction_cols])
            errors = (preds - target) ** 2
            best_idx = np.argmin(errors)

            one_hot_weights = np.zeros(len(prediction_cols))
            one_hot_weights[best_idx] = 1

            # Acumulamos
            if product_id not in product_weights:
                product_weights[product_id] = []
            product_weights[product_id].append(one_hot_weights)

        # Promediamos los pesos por product_id
        for product_id, weight_list in product_weights.items():
            avg_weights = np.mean(weight_list, axis=0)
            weights_dict = dict(zip(prediction_cols, avg_weights))

            # Calculamos la predicción promedio ponderada usando los pesos promedio
            product_rows = results_df[results_df["product_id"] == product_id]
            preds_matrix = product_rows[prediction_cols].values
            weighted_preds = preds_matrix @ avg_weights
            mean_prediction = np.mean(weighted_preds)

            weights_list.append(weights_dict)
            predictions_list.append(mean_prediction)

        # Creamos el DataFrame final
        product_ids = list(product_weights.keys())
        agg_df = pd.DataFrame({
            "product_id": product_ids,
            "weights": weights_list,
            "predictions": predictions_list
        })

        self.model_weights = agg_df[["product_id", "weights"]].set_index("product_id")


    def _compute_metrics_simple(self, y_true, y_pred):
        """Calcula el error absoluto medio entre y_true e y_pred"""
        return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) if np.sum(y_true) > 0 else 0

    def _compute_metrics(self, results_df):
        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]
        agg_df = results_df.groupby("product_id")[["target"] + prediction_cols].sum().reset_index()
        agg_df = agg_df.set_index("product_id")
        agg_df["weights"] = self.model_weights["weights"]
        agg_df["prediction_ensamble"] = agg_df.apply(
            lambda row: sum(row[col] * row["weights"][col] for col in prediction_cols), 
            axis=1
        )
        self.agg_df = agg_df
        # calculo metricas
        metrics = {}
        def total_error(y_true, y_pred):
            return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

        prediction_cols = [col for col in agg_df.columns if col.startswith("prediction_")]
        for col in prediction_cols:
            metrics[col] = total_error(agg_df["target"], agg_df[col])
        return pd.DataFrame(metrics, index=[0]).T.rename(columns={0: "error"})
    
    def fit(self, df, splitter):
        """Entrena todos los modelos en cada split del splitter"""
        df = df.dropna(subset=["target"])
        number_of_splits = splitter.get_n_splits(df)
        results = {f"split_{i}": [] for i in range(number_of_splits)}
        for i, (train_idx, test_idx) in enumerate(splitter.split(df)):
            # Obtener las fechas de los splits originales
            train_dates = df.iloc[train_idx]["date_id"].unique()
            test_dates = df.iloc[test_idx]["date_id"].unique()
            
            for m in self.models:
                model = deepcopy(m)
                df_model = model.prepare_dataset(df)
                
                # Recalcular train/test usando las fechas, no los índices
                train_df = df_model[df_model["date_id"].isin(train_dates)]
                test_df = df_model[df_model["date_id"].isin(test_dates)]
                
                pred_df = model.fit_and_predict(train_df, test_df, df_model)
                results[f"split_{i}"].append({
                    "model": model,
                    "target": test_df[["target", "product_id", "date_id"]],
                    "pred_df": pred_df
                })
                print(f"Modelo {model.name} entrenado en split {i+1}/{number_of_splits}:")
                print(self._compute_metrics_simple(pred_df["target"], pred_df["prediction"]))
        results_df = self._combina_results(results)
        self._optimize_weights(results_df)
        print(self._compute_metrics(results_df))

    def final_pred(self, df, kaggle_date_id):
        """Vuelve a entrenar todos los modelos con el dataset completo"""
        results = {"split_final": []}
        for m in self.models:
            model = deepcopy(m)
            df_model = model.prepare_dataset(df)
            # Recalcular train/test usando las fechas, no los índices
            train_df = df_model[df_model["date_id"] < kaggle_date_id]
            train_df = train_df.dropna(subset=["target"])
            pred_df = df_model[df_model["date_id"] == kaggle_date_id]

            pred_df = model.fit_and_predict(train_df, pred_df, df_model)
            results["split_final"].append({
                "model": model,
                "target": pred_df[["target", "product_id", "date_id"]],
                "pred_df": pred_df
            })
        results_df = self._combina_results(results)  
        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]
        agg_df = results_df.groupby("product_id")[["target"] + prediction_cols].sum().reset_index()
        agg_df = agg_df.set_index("product_id", drop=False)
        agg_df["weights"] = self.model_weights["weights"]
        agg_df["prediction_ensamble"] = agg_df.apply(
            lambda row: sum(row[col] * row["weights"][col] for col in prediction_cols), 
            axis=1
        )
        return agg_df.rename(columns={"prediction_ensamble": "tn"})


In [ ]:
# TODO: entrenar el lightgbm con todos los product_ids? (la validacion solo con los 780)
import sys
from contextlib import redirect_stdout

trainer = EnsambleTrainer([
    LinearRegressionModel(),
    AutoGluonTabularPredictor(presets="best", time_budget=3600, exclude_model_types=["KNN"]),
    #AutoGluonTabularPredictor(presets="good", time_budget=3600, exclude_model_types=["KNN"]),
    AutoGluonPredictor(presets="best_quality"),
    #AutoGluonPredictor(presets="fast_training"),
    SimpleMovingAveragePredictor(window_size=12)
])
splitter = CustomTimeSeriesSplit(n_splits=3, gap=1)
trainer.fit(df, splitter)


Registros de entrenamiento: 33
Modelo LinearRegression entrenado en split 1/1:
0.332429


/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Scaling


No path specified. Models will be saved in: "AutogluonModels/ag-20250717_002833"
Preset alias specified: 'best' maps to 'best_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 26 14:16:59 UTC 2
CPU Count:          16
Memory Avail:       14.92 GB / 31.23 GB (47.8%)
Disk Space Avail:   739.83 GB / 914.78 GB (80.9%)
Presets specified: ['best']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the 

(_ray_fit pid=48249) [1000]	valid_set's l1: 0.366053
(_ray_fit pid=48249) [2000]	valid_set's l1: 0.357647 [repeated 4x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(_ray_fit pid=48251) [2000]	valid_set's l1: 0.351398 [repeated 3x across cluster]


...

(_ray_fit pid=50694) [1000]	valid_set's l1: 0.355463
(_ray_fit pid=50698) [1000]	valid_set's l1: 0.361884
(_ray_fit pid=50694) [2000]	valid_set's l1: 0.347146 [repeated 3x across cluster]


(_ray_fit pid=50694) 	Ran out of time, early stopping on iteration 2660. Best iteration is: [repeated 4x across cluster]
(_ray_fit pid=50694) 	[2660]	valid_set's l1: 0.343925 [repeated 4x across cluster]
(_dystack pid=46391) 	-0.3526	 = Validation score   (-mean_absolute_error)
(_dystack pid=46391) 	474.78s	 = Training   runtime
(_dystack pid=46391) 	6.49s	 = Validation runtime
(_dystack pid=46391) Fitting model: LightGBM_BAG_L1 ... Training model for up to 111.23s of the 406.57s of remaining time.
(_dystack pid=46391) 	Memory not enough to fit 8 folds in parallel. Will train 4 folds in parallel instead (Estimated 11.15% memory usage per fold, 44.60%/80.00% total).
(_dystack pid=46391) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (4 workers, per: cpus=4, gpus=0, memory=11.15%)
(_ray_fit pid=52774) 	Ran out of time, early stopping on iteration 268. Best iteration is: [repeated 4x across cluster]
(_ray_fit pid=52774) 	[268]	valid_set's l1: 0.38613

Modelo AutoGluonTabular entrenado en split 1/1:
0.28506225


No path specified. Models will be saved in: "AutogluonModels/ag-20250717_012851"
Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250717_012851'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 26 14:16:59 UTC 2
CPU Count:          16
GPU Count:          1
Memory Avail:       17.76 GB / 31.23 GB (56.9%)
Disk Space Avail:   739.20 GB / 914.78 GB (80.8%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 't

Modelo AutoGluon-best_quality entrenado en split 1/1:
0.22939986008175528
Modelo SMA-12 entrenado en split 1/1:
0.2922211481575493
                                      error
prediction_AutoGluon-best_quality  0.233124
prediction_AutoGluonTabular        0.285062
prediction_LinearRegression        0.333704
prediction_SMA-12                  0.292221
prediction_ensamble                0.149960


In [16]:
trainer.model_weights

,weights
product_id,
20001.0,"{'prediction_AutoGluon-best_quality': 0.0, 'pr..."
20002.0,"{'prediction_AutoGluon-best_quality': 0.0, 'pr..."
20003.0,"{'prediction_AutoGluon-best_quality': 1.0, 'pr..."
20004.0,"{'prediction_AutoGluon-best_quality': 0.0, 'pr..."
20005.0,"{'prediction_AutoGluon-best_quality': 0.0, 'pr..."
...,...
21252.0,"{'prediction_AutoGluon-best_quality': 1.0, 'pr..."
21265.0,"{'prediction_AutoGluon-best_quality': 0.0, 'pr..."
21266.0,"{'prediction_AutoGluon-best_quality': 0.0, 'pr..."


In [17]:
trainer._compute_metrics(trainer.train_results)

,error
prediction_AutoGluon-best_quality,0.233124
prediction_AutoGluonTabular,0.285062
prediction_LinearRegression,0.333704
prediction_SMA-12,0.292221
prediction_ensamble,0.149960


In [18]:
trainer.train_results[trainer.train_results["product_id"] == 20001.0].head(10)

,product_id,date_id,target,prediction_AutoGluon-best_quality,prediction_AutoGluonTabular,prediction_LinearRegression,prediction_SMA-12
0,20001.0,33.0,1504.688477,1371.509513,1407.251221,1195.375244,1487.869466


In [19]:

models_used = [model.name for model in trainer.models]
models_used = " ".join(models_used)
# hago un hash en base de models_used para el nombre del archivo
import hashlib
hash_object = hashlib.md5(models_used.encode())
hash_hex = hash_object.hexdigest()

trainer.train_results.to_csv(f"train_results_{hash_hex}.csv", index=False)

In [20]:
trainer.agg_df

,target,prediction_AutoGluon-best_quality,prediction_AutoGluonTabular,prediction_LinearRegression,prediction_SMA-12,weights,prediction_ensamble
product_id,,,,,,,
20001.0,1504.688477,1371.509513,1407.251221,1195.375244,1487.869466,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1487.869466
20002.0,1087.308594,1051.628792,1075.765137,1305.163086,1197.552099,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1075.765137
20003.0,892.501282,845.978727,805.669800,668.048706,796.305669,"{'prediction_AutoGluon-best_quality': 1.0, 'pr...",845.978727
20004.0,637.900024,701.515587,635.014221,632.832642,629.387779,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",635.014221
20005.0,593.244446,643.428391,567.499268,738.727234,638.415230,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",567.499268
...,...,...,...,...,...,...,...
21263.0,0.012700,-0.000660,0.008420,0.000000,0.063882,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.008420
21265.0,0.050070,0.067998,0.007710,0.057709,0.097417,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.057709
21266.0,0.051210,0.083060,0.007755,0.064782,0.103531,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.064782


In [43]:
agg_df = trainer.agg_df.copy()
pred_columns = [col for col in agg_df.columns if col.startswith("prediction_") if col != "prediction_ensamble"]

# entreno un linear regression para usar las predicciones y hacer una prediccion final
from sklearn.linear_model import LinearRegression
X = agg_df[pred_columns]
y = agg_df["target"]
model = LinearRegression()
model.fit(X, y)
agg_df["prediction_final"] = model.predict(X)
total_error = np.sum(np.abs(agg_df["target"] - agg_df["prediction_final"])) / np.sum(agg_df["target"])
print(f"Total error: {total_error}")

# calculo weights para cada product_id usando linear regression en lugar de scipy.minimize
from scipy.optimize import minimize

def optimize_weights_per_product(y_true, y_pred_values):
    """Optimiza los pesos para UNA SOLA fila usando scipy.minimize"""
    predictions = np.array(y_pred_values)
    
    if np.allclose(predictions, 0) or y_true == 0:
        # Si todas las predicciones son 0 o target es 0, usar pesos uniformes
        return np.ones(len(predictions)) / len(predictions)
    
    def objective(weights):
        """Función objetivo: error absoluto"""
        weighted_pred = np.dot(predictions, weights)
        return abs(y_true - weighted_pred)
    
    # Restricciones
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}  # Suma = 1
    bounds = [(0, 1) for _ in range(len(predictions))]  # Pesos entre 0 y 1
    initial_weights = np.ones(len(predictions)) / len(predictions)  # Pesos iniciales uniformes
    
    try:
        result = minimize(objective, initial_weights, method='SLSQP', 
                         bounds=bounds, constraints=constraints)
        if result.success:
            return result.x
        else:
            return initial_weights
    except:
        return initial_weights

# Aplicar la optimización
agg_df["weights_2"] = agg_df.apply(
    lambda row: optimize_weights_per_product(row["target"], row[pred_columns].values), 
    axis=1
)

# Calcular la predicción ponderada
agg_df["prediction_final_2"] = agg_df.apply(
    lambda row: np.dot(row[pred_columns].values, row["weights_2"]), 
    axis=1
)

total_error_2 = np.sum(np.abs(agg_df["target"] - agg_df["prediction_final_2"])) / np.sum(agg_df["target"])
print(f"Total error with scipy optimized weights: {total_error_2}")
agg_df

Total error: 0.2491763197322286
Total error with scipy optimized weights: 0.1292758573993901


,target,prediction_AutoGluon-best_quality,prediction_AutoGluonTabular,prediction_LinearRegression,prediction_SMA-12,weights,prediction_ensamble,prediction_final,weights_2,prediction_final_2
product_id,,,,,,,,,,
20001.0,1504.688477,1371.509513,1407.251221,1195.375244,1487.869466,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1487.869466,1376.151307,"[0.0, 0.0, 1.0375585409696145e-10, 0.999999999...",1487.869466
20002.0,1087.308594,1051.628792,1075.765137,1305.163086,1197.552099,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1075.765137,999.124000,"[0.7472919870624402, 0.08424499720739709, 0.08...",1087.308593
20003.0,892.501282,845.978727,805.669800,668.048706,796.305669,"{'prediction_AutoGluon-best_quality': 1.0, 'pr...",845.978727,827.780079,"[0.9999999994278128, 0.0, 9.636755005093534e-1...",845.978727
20004.0,637.900024,701.515587,635.014221,632.832642,629.387779,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",635.014221,665.231499,"[0.10483188100656335, 0.10483107742450411, 0.1...",637.900027
20005.0,593.244446,643.428391,567.499268,738.727234,638.415230,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",567.499268,605.637898,"[0.08094097375590485, 0.7571761005569998, 0.08...",593.244438
...,...,...,...,...,...,...,...,...,...,...
21263.0,0.012700,-0.000660,0.008420,0.000000,0.063882,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.008420,-2.469173,"[0.2837836605451781, 0.26710843645884846, 0.28...",0.012700
21265.0,0.050070,0.067998,0.007710,0.057709,0.097417,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.057709,-2.399756,"[0.2312075588033872, 0.34131479667586584, 0.24...",0.050070
21266.0,0.051210,0.083060,0.007755,0.064782,0.103531,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.064782,-2.383936,"[0.20124145655012463, 0.4021255666922487, 0.25...",0.051210


In [21]:
from contextlib import redirect_stdout

with open("entrenamiento_final.log", "w") as f:
    with redirect_stdout(f):
        final_df = trainer.final_pred(df, 35)

No path specified. Models will be saved in: "AutogluonModels/ag-20250717_014435"
Preset alias specified: 'best' maps to 'best_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 26 14:16:59 UTC 2
CPU Count:          16
Memory Avail:       14.25 GB / 31.23 GB (45.6%)
Disk Space Avail:   738.59 GB / 914.78 GB (80.7%)
Presets specified: ['best']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the 

In [22]:
final_df

,product_id,target,prediction_AutoGluon-best_quality,prediction_AutoGluonTabular,prediction_LinearRegression,prediction_SMA-12,weights,tn
product_id,,,,,,,,
20001.0,20001.0,0.0,1299.510426,1482.286499,1162.707275,1454.732707,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1454.732707
20002.0,20002.0,0.0,1045.332183,1305.831787,1183.640381,1175.437159,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1305.831787
20003.0,20003.0,0.0,753.440715,971.087341,684.763855,784.976400,"{'prediction_AutoGluon-best_quality': 1.0, 'pr...",753.440715
20004.0,20004.0,0.0,564.919682,711.953064,580.484985,627.215335,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",711.953064
20005.0,20005.0,0.0,553.520154,662.820068,563.560791,668.270103,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",662.820068
...,...,...,...,...,...,...,...,...
21263.0,21263.0,0.0,0.016636,0.005712,0.467749,0.029993,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.005712
21265.0,21265.0,0.0,0.051505,0.006018,0.049021,0.089541,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.049021
21266.0,21266.0,0.0,0.056870,0.006136,0.052555,0.094659,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.052555


In [48]:
final_df_2 = final_df.copy()
final_df_2["pred_linear"] = model.predict(final_df_2[pred_columns])
final_df_2["pred_linear"] = final_df_2["pred_linear"].clip(lower=0)  # Aseguro que la prediccion no sea negativa
final_df_2

final_df_3 = final_df_2.copy()
final_df_3["weights_2"] = agg_df["weights_2"]
final_df_3["pred_weights"] = final_df_3.apply(
    lambda row: np.dot(row[pred_columns].values, row["weights_2"]),
    axis=1
)
final_df_3["pred_weights"] = final_df_3["pred_weights"].clip(lower=0)  # Aseguro que la prediccion no sea negativa
final_df_3


,product_id,target,prediction_AutoGluon-best_quality,prediction_AutoGluonTabular,prediction_LinearRegression,prediction_SMA-12,weights,tn,pred_linear,weights_2,pred_weights
product_id,,,,,,,,,,,
20001.0,20001.0,0.0,1299.510426,1482.286499,1162.707275,1454.732707,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1454.732707,1281.490046,"[0.0, 0.0, 1.0375585409696145e-10, 0.999999999...",1454.732707
20002.0,20002.0,0.0,1045.332183,1305.831787,1183.640381,1175.437159,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1305.831787,956.038119,"[0.7472919870624402, 0.08424499720739709, 0.08...",1089.886781
20003.0,20003.0,0.0,753.440715,971.087341,684.763855,784.976400,"{'prediction_AutoGluon-best_quality': 1.0, 'pr...",753.440715,690.868024,"[0.9999999994278128, 0.0, 9.636755005093534e-1...",753.440716
20004.0,20004.0,0.0,564.919682,711.953064,580.484985,627.215335,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",711.953064,522.895708,"[0.10483188100656335, 0.10483107742450411, 0.1...",624.669119
20005.0,20005.0,0.0,553.520154,662.820068,563.560791,668.270103,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",662.820068,542.749539,"[0.08094097375590485, 0.7571761005569998, 0.08...",646.380264
...,...,...,...,...,...,...,...,...,...,...,...
21263.0,21263.0,0.0,0.016636,0.005712,0.467749,0.029993,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.005712,0.000000,"[0.2837836605451781, 0.26710843645884846, 0.28...",0.143414
21265.0,21265.0,0.0,0.051505,0.006018,0.049021,0.089541,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.049021,0.000000,"[0.2312075588033872, 0.34131479667586584, 0.24...",0.042109
21266.0,21266.0,0.0,0.056870,0.006136,0.052555,0.094659,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.052555,0.000000,"[0.20124145655012463, 0.4021255666922487, 0.25...",0.040931


In [28]:
df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "tn_rolling_mean_12": "sum"})

,product_id,date_id,tn,tn_rolling_mean_12
0,20001,0,934.772217,0.0
1,20001,1,798.016174,0.0
2,20001,2,1303.357666,0.0
3,20001,3,1069.961304,0.0
4,20001,4,1502.201416,0.0
...,...,...,...,...
22370,21276,31,0.012650,0.0
22371,21276,32,0.018560,0.0
22372,21276,33,0.020790,0.0
22373,21276,34,0.033410,0.0


In [ ]:

models_used = [model.name for model in trainer.models]
models_used = " ".join(models_used)
# hago un hash en base de models_used para el nombre del archivo
import hashlib
hash_object = hashlib.md5(models_used.encode())
hash_hex = hash_object.hexdigest()
description = f"Ensamble de modelos: {models_used}"

submission = final_df[["product_id", "tn"]].reset_index(drop=True)
submission.to_csv(f"submission_weighted_ensamble_{hash_hex}.csv", index=False)
submission

,product_id,tn
0,20001.0,1454.732707
1,20002.0,1305.831787
2,20003.0,753.440715
3,20004.0,711.953064
4,20005.0,662.820068
...,...,...
775,21263.0,0.005712
776,21265.0,0.049021
777,21266.0,0.052555
778,21267.0,0.004600


In [38]:
submission_2 = final_df_2[["product_id", "pred_linear"]].rename(columns={"pred_linear": "tn"}).reset_index(drop=True)
submission_2.to_csv(f"submission_linear_regression_{hash_hex}.csv", index=False)
submission_2

,product_id,tn
0,20001.0,1281.490046
1,20002.0,956.038119
2,20003.0,690.868024
3,20004.0,522.895708
4,20005.0,542.749539
...,...,...
775,21263.0,0.000000
776,21265.0,0.000000
777,21266.0,0.000000
778,21267.0,0.000000


In [24]:
print(description)

Ensamble de modelos: LinearRegression AutoGluonTabular AutoGluon-best_quality SMA-12


In [49]:
submission_3 = final_df_3[["product_id", "pred_weights"]].rename(columns={"pred_weights": "tn"}).reset_index(drop=True)
submission_3.to_csv(f"submission_weights_ensamble_per_product_{hash_hex}.csv", index=False)
submission_3

,product_id,tn
0,20001.0,1454.732707
1,20002.0,1089.886781
2,20003.0,753.440716
3,20004.0,624.669119
4,20005.0,646.380264
...,...,...
775,21263.0,0.143414
776,21265.0,0.042109
777,21266.0,0.040931
778,21267.0,0.010882
